# DSA Drill — Binary Tree Maximum Path Sum (LC 124 · Hard)

> A **path** is any node-to-node sequence (each node once, adjacent steps). Return the **maximum path sum**. Values may be negative; the path must contain **at least one node**.

**Examples**
1. `[1,2,3]` → 6 (2→1→3)
2. `[-10, 9, 20, 15, 7]` → 42 (15→20→7)
3. `[-3]` → **-3** — the path must contain one node

**The key distinction (verbalize cold):** what the parent *receives* is a **straight-line gain** (at most ONE child chain) — but the answer may use a **bend**: left-chain + me + right-chain. Every node computes both; only the bend updates the global max.

**Landmines**
1. Why must the value *returned upward* be `me + max(left_gain, right_gain, 0)` and never the bend?
2. Why `max(gain, 0)` — and which example forces the answer to be a *single negative node*?
3. Where does the global answer live, and why?

One hint allowed.

## Intuition (plain): every path has exactly one bend

Look at any optimal path: it goes **up** from some deep node, passes through exactly one **peak**, and comes back **down** — one bend. So each node answers two different questions:

- **What can I offer my parent?** A straight chain through me: `me + max(left_gain, right_gain, 0)` — a path going *upward* cannot fork through two children.
- **What is the best path bending at ME?** `me + left_gain + right_gain` (gains clamped at 0) — the fork is legal *here*.

The bend updates a global max; the straight chain returns upward. The global max is the answer.

In [ ]:
from collections import namedtuple

TNode = namedtuple('TNode', 'val left right', defaults=(None, None))


def max_path_sum(root) -> float:
    """LC 124: maximum path sum, single pass, O(n) nodes / O(h) stack."""
    best = float('-inf')

    def gains(node):
        nonlocal best
        if node is None:
            return 0.0
        left_gain = max(gains(node.left), 0.0)       # prune negative branches
        right_gain = max(gains(node.right), 0.0)
        bend = node.val + left_gain + right_gain     # best path bending at me
        best = max(best, bend)
        return node.val + max(left_gain, right_gain) # straight chain for my parent

    gains(root)
    return best

In [ ]:
class MNode:
    """MUTABLE tree node (namedtuples reject attribute assignment)."""
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree(level_list):
    """Heap-indexed format: node i has children 2i+1 and 2i+2 (None = missing)."""
    def build(i):
        if i >= len(level_list) or level_list[i] is None:
            return None
        return MNode(level_list[i], build(2 * i + 1), build(2 * i + 2))
    return build(0)

assert max_path_sum(build_tree([1, 2, 3])) == 6
assert max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])) == 42  # heap-index format
assert max_path_sum(build_tree([-3])) == -3

# Fuzz: reference = enumerate EVERY simple path via LCA join (O(n^2) pairs, fine for n<=9)
import random, itertools
random.seed(13)

def brute(root):
    nodes = []
    def collect(n):
        if n:
            nodes.append(n); collect(n.left); collect(n.right)
    collect(root)
    def find_lca(n, p, q):
        if n is None or n is p or n is q:
            return n
        l, r = find_lca(n.left, p, q), find_lca(n.right, p, q)
        return n if (l and r) else (l or r)
    def chain(n, target, acc):
        if n is None:
            return None
        if n is target:
            return acc + n.val
        opts = [chain(n.left, target, acc + n.val), chain(n.right, target, acc + n.val)]
        return max((o for o in opts if o is not None), default=None)
    best = max(n.val for n in nodes)
    for a, b in itertools.combinations_with_replacement(nodes, 2):
        lca = find_lca(root, a, b)
        total = chain(lca, a, 0) + chain(lca, b, 0) - lca.val
        best = max(best, total)
    return best

for _ in range(60):
    n = random.randint(1, 9)
    vals = [random.randint(-10, 10) for _ in range(n)]
    got = max_path_sum(build_tree(vals))
    exp = brute(build_tree(vals))
    assert got == exp, (vals, got, exp)
print("examples + 60 random trees vs all-paths reference: all agree")

In [ ]:
# Edge cases - run me, say WHY out loud
assert max_path_sum(MNode(-3)) == -3                       # single NEGATIVE node still counts
assert max_path_sum(build_tree([1, -2, -3])) == 1          # children pruned; answer is root alone
assert max_path_sum(build_tree([2, -1, -1, 8, 8, 8, 8])) == 16  # 8+(-1)+2+(-1)+8: the CROSS-bend path (bend at the root)
print("edge cases pass OK")

## Post-mortem (fill in your own words)

- Straight-chain vs bend — why the parent only gets a straight chain:
- Why `max(gain, 0)` pruning matters (which example):
- Where the global answer lives and why `nonlocal` is needed:
- Complexity:
- One-line pattern (copy into `dsa/patterns.md`):